# RecruitAI Evaluation

Evaluate how well the multi-agent system chooses **continue**, **schedule**, or **end** on the labeled recruiter turns.

Metrics follow Lesson 17: `accuracy_score` and `confusion_matrix`.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.metrics import accuracy_score, confusion_matrix

ROOT = Path.cwd().parent if Path.cwd().name == "tests" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from app.modules.agents import exit_advisor
from app.modules.agents.orchestrator import predict_action
from app.modules.finetuning.prepare_data import build_labeled_rows

df = build_labeled_rows()
print(df.shape)
print(df["label"].value_counts())
df.head()

## Exit Advisor only (sklearn, no API)

The Exit Advisor is trained to predict `end` vs `not_end`.

In [ ]:
exit_preds = [exit_advisor.predict(text)["decision"] for text in df["history"]]
exit_true = df["end_label"].tolist()
exit_pred_labels = ["end" if value == "end" else "not_end" for value in exit_preds]

print("Exit accuracy:", accuracy_score(exit_true, exit_pred_labels))
cf_exit = confusion_matrix(exit_true, exit_pred_labels, labels=["end", "not_end"])
sns.heatmap(cf_exit, annot=True, fmt="d", cmap="Blues",
            xticklabels=["end", "not_end"], yticklabels=["end", "not_end"])
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Exit Advisor confusion matrix")
plt.show()

## Full action evaluation (needs OPENAI_API_KEY)

For each labeled recruiter turn, build the history before that turn and ask the orchestrator to vote:

- Exit Advisor first
- If not ending, Scheduling Advisor
- Priority: end > schedule > continue

In [ ]:
predictions = []
for row in df.itertuples(index=False):
    result = predict_action(row.history, conversation_dt=row.timestamp_utc)
    predictions.append(result["action"])
    print(row.conversation_id, row.turn_id, row.label, "->", result["action"])

df = df.copy()
df["predicted"] = predictions
print("\nAccuracy:", accuracy_score(df["label"], df["predicted"]))
df[["conversation_id", "turn_id", "label", "predicted"]].head(10)

In [ ]:
labels = ["continue", "schedule", "end"]
cf = confusion_matrix(df["label"], df["predicted"], labels=labels)
print(cf)

sns.heatmap(cf, annot=True, fmt="d", cmap="Blues", xticklabels=labels, yticklabels=labels)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Continue / Schedule / End confusion matrix")
plt.show()

## Notes

- Labels exist only on recruiter turns.
- The class counts are small and imbalanced.
- Evaluation scores the **action**, not the exact SMS wording.
- Monday requests cannot match the seed calendar because the seed has no Mondays.